# GPT-2 Baseline 架构

源码导航：[`core/model/gpt2.py`](../../../core/model/gpt2.py)。

## 1. 理论背景

### Pre-LN 残差块

GPT-2 采用 **Pre-LayerNorm** 结构，即 LayerNorm 作用在残差路径输入之前，与 Transformer 原论文的 Post-LN 不同：

$$x \leftarrow x + \text{Attn}(\text{LN}_1(x)), \quad x \leftarrow x + \text{MLP}(\text{LN}_2(x))$$

Pre-LN 使梯度可以不经过 LN 直接回传到靠近输入的层，训练稳定性优于 Post-LN，是 GPT-2/3 系列的重要工程改进。

### 数据流

```
idx: (B, T)
  → wte: (B, T, d)      # token embedding
  → wpe: (T, d)          # learned positional embedding（与 idx 广播相加）
  → Dropout
  → Block × N            # Pre-LN 残差块
  → ln_f                 # final LayerNorm
  → lm_head (Linear, no bias)  # (B, T, V) 或推理时 (B, 1, V)
```

### 权重共享（Weight Tying）

输入嵌入矩阵 $W_E \in \mathbb{R}^{V \times d}$（`wte.weight`）与输出投影矩阵 $W_U \in \mathbb{R}^{V \times d}$（`lm_head.weight`）共享同一张量：

$$\text{lm\_head.weight} = \text{wte.weight}$$

参数节省 $V \times d$（对 GPT-2-124M 为 $50257 \times 768 \approx 38.6\text{M}$）。反向传播时两处梯度在同一张量上累加，相当于对 embedding 矩阵施加了正则约束。

### 残差路径输出投影的初始化缩放

GPT-2 论文对 `c_proj`（MHA 输出投影）和 MLP 输出层（`fc_proj`）的权重初始化标准差按层数缩放：

$$\sigma_{\text{c\_proj}} = \frac{\sigma_0}{\sqrt{2N}}$$

其中 $N$ 为 Transformer 块总数。N 层残差块的贡献方差之和为 $N \cdot \sigma_0^2 / (2N) = \sigma_0^2 / 2$，防止深层网络初始化时信号方差爆炸。

### GPT-2 与现代 LLM 的结构对比

| 组件 | GPT-2 | Walkie / LLaMA 系列 |
|---|---|---|
| 位置编码 | Learned Absolute PE | RoPE |
| 归一化 | LayerNorm（含 bias） | RMSNorm（无 bias） |
| 注意力 | MHA（全头） | GQA + QK-Norm |
| FFN 激活 | GELU | SwiGLU |
| 权重 bias | 全部 True | 全部 False |

In [ ]:
from __future__ import annotations

import sys
import math
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.model.gpt2 import GPT2Config, GPT2LMHeadModel

### 2. Sanity Check：前向传播、损失与参数统计

In [ ]:
cfg = GPT2Config(
    vocab_size=50257, n_layer=2, n_head=2, n_embd=64,
    block_size=32, dropout=0.0, bias=True, tie_weights=True,
)
model = GPT2LMHeadModel(cfg)

torch.manual_seed(0)
B, T = 2, 16
idx     = torch.randint(0, cfg.vocab_size, (B, T))
targets = torch.randint(0, cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f"logits shape : {tuple(logits.shape)}")   # (B, T, V)
print(f"cross-entropy loss : {loss.item():.4f}  (untrained baseline ≈ ln(V)={math.log(cfg.vocab_size):.4f})")

# 权重共享验证
assert model.lm_head.weight.data_ptr() == model.wte.weight.data_ptr(), \
    "lm_head 与 wte 权重未共享！"
print(f"\n权重共享已验证: wte.weight data_ptr = lm_head.weight data_ptr")

# 参数量统计（不计入 lm_head 因为 weight tying）
n_params = sum(p.numel() for p in model.parameters())
n_unique = sum(p.numel() for n, p in model.named_parameters()
               if not n.startswith('lm_head'))
print(f"\n总参数（含共享）: {n_params:,}")
print(f"唯一参数（不重计 lm_head）: {n_unique:,}")

# c_proj 权重初始化缩放验证（std ≈ init_std / sqrt(2*N)）
expected_std = cfg.init_std / math.sqrt(2 * cfg.n_layer)
for n, p in model.named_parameters():
    if 'c_proj.weight' in n:
        print(f"{n}: std={p.std().item():.4f}  (expected ≈ {expected_std:.4f})")

### 3. 源码精讲

**`Block.forward`**（Pre-LN 双残差结构）：

```python
def forward(self, x):
    # 第一个子层：Pre-LN + 因果自注意力 + 残差
    x = x + self.attn(self.ln_1(x))
    # 第二个子层：Pre-LN + GELU MLP + 残差
    x = x + self.mlp(self.ln_2(x))
    return x
```

**`GPT2LMHeadModel.__init__`**（权重共享 + 残差路径缩放初始化）：

```python
# 输入嵌入
self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)   # token embedding
self.wpe = LearnedPositionalEmbedding(cfg.block_size, cfg.n_embd)  # learned absolute PE

# 语言建模头（无 bias）
self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

# Weight Tying：共享同一 Parameter 对象（非拷贝）
if cfg.tie_weights:
    self.lm_head.weight = self.wte.weight

# 全局初始化后，对残差路径输出投影单独按层数缩放
for pn, p in self.named_parameters():
    if pn.endswith('c_proj.weight'):
        nn.init.normal_(p, mean=0.0, std=cfg.init_std / math.sqrt(2 * cfg.n_layer))
```

**`GPT2LMHeadModel.forward`**（训练 vs 推理分支）：

```python
tok_emb = self.wte(idx)          # (B, T, d)
pos_emb = self.wpe(T, ...)       # (T, d)  LearnedPositionalEmbedding 返回绝对位置向量
x = self.drop(tok_emb + pos_emb)
for blk in self.h:
    x = blk(x)
x = self.ln_f(x)

if targets is not None:
    logits = self.lm_head(x)           # (B, T, V)  完整序列 logits，用于计算交叉熵
    loss = F.cross_entropy(logits.view(-1, V), targets.view(-1), ignore_index=-1)
    return logits, loss
else:
    logits = self.lm_head(x[:, -1:, :])  # 推理时只算最后一步，节省 O(T*V) 计算
    return logits, None
```

---

## 延伸阅读与参考资料

### 核心论文
- **GPT-2 技术报告**: Radford et al., 2019. *Language Models are Unsupervised Multitask Learners*. [PDF](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- **Attention Is All You Need**: Vaswani et al., 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)

### 参考实现与资源
- **nanoGPT**: Karpathy, 2022. [GitHub](https://github.com/karpathy/nanoGPT)
- **The Illustrated GPT-2**: Jay Alammar. [Link](https://jalammar.github.io/illustrated-gpt2/)